# Quant Skills — Lesson 1: Expected Value, Your Edge, and Simulation

**Goal of this lesson:** learn the three tools every quant uses every day —

1. **Expected value (EV)** — is a bet worth taking?
2. **Trade statistics** — what does *your* track record actually say?
3. **Monte Carlo simulation** — what range of outcomes should you expect?

**How to use this notebook:** click a code cell, then press **Shift + Enter** to run it. Run the cells top to bottom.
Change the numbers in any cell and run it again — that's how you learn.

## Part 1 — Expected value

Every trade, every market-making quote, every interview puzzle comes back to one formula:

**EV = (probability of win × amount won) − (probability of loss × amount lost)**

If EV is positive, the bet makes money *over many repetitions*. It says nothing about any single bet.

In [ ]:
def expected_value(p_win, win_amount, loss_amount):
    """Average profit per bet over many repetitions."""
    return p_win * win_amount - (1 - p_win) * loss_amount

# A debit spread from your plan: 45% win rate, +$165 winners, -$85 losers
print("Debit spread EV per spread:", round(expected_value(0.45, 165, 85), 2))

# A credit spread with a nice win rate but a big loss when wrong
print("Credit spread EV per spread:", round(expected_value(0.80, 62.50, 300), 2))

**Notice:** the credit spread wins 80% of the time and still has *negative* EV with those numbers.
Win rate alone tells you nothing. The size of the win versus the loss matters just as much.

### Breakeven win rate

For a trade that wins `R` times what it risks, the win rate where EV = 0 is:

**breakeven = 1 / (1 + R)**

In [ ]:
def breakeven_win_rate(reward_to_risk):
    return 1 / (1 + reward_to_risk)

for R in [0.5, 1.0, 1.5, 1.67, 2.0, 3.0]:
    print(f"Win {R}R per 1R risked -> need {breakeven_win_rate(R):.1%} win rate to break even")

## Part 2 — Your own track record

These are your actual daily results from 10/16/2026 to 10/26/2026 on the $50,000 compounding account.
Every time you add days to your Google Sheet, add them to this list too and re-run.

In [ ]:
import numpy as np

start_balance = 50_000
daily_gains = [57.92, 65.80, 54.80, 97.80, 164.00, -183.96, 26.45]   # your real results

# Build the equity curve and daily % returns
balances = [start_balance]
for g in daily_gains:
    balances.append(balances[-1] + g)
balances = np.array(balances)
returns = np.diff(balances) / balances[:-1]

days = len(returns)
wins = returns[returns > 0]
losses = returns[returns < 0]

avg_daily = (balances[-1] / start_balance) ** (1 / days) - 1          # compounded
vol_daily = returns.std(ddof=1)
vol_annual = vol_daily * np.sqrt(252)
risk_free = 0.04
sharpe = (np.mean(returns) * 252 - risk_free) / vol_annual
peak = np.maximum.accumulate(balances)
max_drawdown = ((balances - peak) / peak).min()

print(f"Trading days:            {days}")
print(f"Ending balance:          ${balances[-1]:,.2f}")
print(f"Avg daily return:        {avg_daily:.3%}   (target 0.250%)")
print(f"Green days:              {len(wins)} of {days}  ({len(wins)/days:.0%})")
print(f"Avg green day:           {wins.mean():.3%}")
print(f"Avg red day:             {losses.mean():.3%}" if len(losses) else "Avg red day: none")
print(f"Daily volatility:        {vol_daily:.3%}")
print(f"Annualized volatility:   {vol_annual:.1%}")
print(f"Sharpe ratio (annual):   {sharpe:.2f}")
print(f"Max drawdown:            {max_drawdown:.2%}")

### What these numbers mean

- **Volatility** — how much your daily results swing. Lower is smoother.
- **Sharpe ratio** — return per unit of volatility. Above 1 is good, above 2 is excellent, above 3 is rare.
- **Max drawdown** — the worst peak-to-trough drop so far.

**Honest warning:** 7 days is far too few to trust any of these. Statistics only become meaningful after ~30–60 observations.
That's exactly why the next tool exists.

## Part 3 — Monte Carlo simulation

You can't wait five years to see if a plan works. So you **simulate it thousands of times** with randomness,
and look at the whole range of outcomes — good luck, bad luck, and everything between.

This simulates your plan: **1% risk per trade, 12 trades a month, 58% win rate, 1.5R winners** — the
"12 trades, 58%" row we worked out.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate(start=50_000, risk=0.01, p_win=0.58, win_R=1.5,
             trades_per_year=144, years=1, n_paths=10_000, seed=42):
    rng = np.random.default_rng(seed)
    n_trades = trades_per_year * years
    outcomes = rng.random((n_paths, n_trades)) < p_win
    multipliers = np.where(outcomes, 1 + risk * win_R, 1 - risk)
    equity = start * np.cumprod(multipliers, axis=1)
    equity = np.hstack([np.full((n_paths, 1), start), equity])
    peak = np.maximum.accumulate(equity, axis=1)
    max_dd = ((equity - peak) / peak).min(axis=1)
    return equity, max_dd

equity, max_dd = simulate()
final = equity[:, -1]

print(f"After 1 year, across 10,000 simulated paths:")
print(f"  Worst 5%:   ${np.percentile(final, 5):,.0f}")
print(f"  Median:     ${np.median(final):,.0f}")
print(f"  Best 5%:    ${np.percentile(final, 95):,.0f}")
print(f"  Chance of ending the year up:        {(final > 50_000).mean():.1%}")
print(f"  Chance of a 10%+ drawdown at some point: {(max_dd <= -0.10).mean():.1%}")
print(f"  Chance of a 20%+ drawdown at some point: {(max_dd <= -0.20).mean():.1%}")

plt.figure(figsize=(9, 4.5))
for i in range(60):
    plt.plot(equity[i], linewidth=0.8, alpha=0.5)
plt.axhline(50_000, color="black", linewidth=1)
plt.title("60 simulated years of the plan (each line = one possible future)")
plt.xlabel("Trade number"); plt.ylabel("Account balance ($)")
plt.show()

## Part 4 — Your exercises

Change one number at a time in the cell below and run it. Write down what happens to the **median** and the **chance of a 20% drawdown**.

1. Drop the win rate to **0.50**. Then **0.45**.
2. Raise risk per trade from **0.01** to **0.03**. Then **0.05**. Then **0.20** (what you were doing before).
3. Keep 0.58 win rate but drop winners to **1.0R**.

The lesson you should discover on your own: **risk size changes the drawdowns far more than it changes the median.**

In [ ]:
equity, max_dd = simulate(p_win=0.58, risk=0.01, win_R=1.5)   # <- change these
final = equity[:, -1]
print(f"Median after 1 year:        ${np.median(final):,.0f}")
print(f"Worst 5% after 1 year:      ${np.percentile(final, 5):,.0f}")
print(f"Chance of 20%+ drawdown:    {(max_dd <= -0.20).mean():.1%}")
print(f"Chance of 50%+ drawdown:    {(max_dd <= -0.50).mean():.1%}")

## Daily mental math (10 minutes)

Firms test speed under time pressure. Practice every day:

- Multiply two 2-digit numbers in your head: 47 × 38, 63 × 29, 84 × 17
- Percentages: 7% of 3,450 · 0.25% of 50,282 · 15% of 728
- Fractions to decimals: 3/8, 5/16, 7/12
- Square roots for volatility: √252 ≈ 15.87, √21 ≈ 4.58, √5 ≈ 2.24

**Next lesson:** pulling real price data with Python and measuring volatility the way desks do.